In [17]:
import subprocess, sys
try:
    import sentence_transformers
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
    import sentence_transformers

In [18]:
import pandas as pd
import numpy as np
import torch
import random
import re
import time
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("torch cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

torch cuda: True
gpu: Tesla T4


In [19]:
# reseeding code to keeps embedding and generation reproducible
SEED = 42

def set_seed(s=SEED):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
print("seed set to", SEED)

seed set to 42


In [20]:
# load data
COMP = "/kaggle/input/competitions/smart-mcq-solver-challenge"
train = pd.read_csv(f"{COMP}/train.csv")
test = pd.read_csv(f"{COMP}/test.csv")
sample = pd.read_csv(f"{COMP}/sample_submission.csv")
OPTIONS = ["A", "B", "C", "D", "E"]
print(f"train: {train.shape}, test: {test.shape}")
train.head(2)

train: (2000, 8), test: (500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A


## Corpus Build

One document per train row with the question and its correct answer text. Only the question is used for retrieval, the answer text is kept to show the model as context.

In [21]:
# build train documents
train_docs = []
for _, r in train.iterrows():
    correct_text = str(r[r["answer"]])
    train_docs.append({
        "prompt": str(r["prompt"]),
        "answer_letter": r["answer"],
        "correct_text": correct_text,
    })

print(f"built {len(train_docs)} documents")
print("sample doc 0:")
print("  Q:", train_docs[0]["prompt"][:120])
print("  A:", train_docs[0]["correct_text"][:120])

built 2000 documents
sample doc 0:
  Q: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? amo
  A: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is 


In [22]:
# encode train prompts with sentence-transformers
from sentence_transformers import SentenceTransformer

set_seed()
device = "cuda" if torch.cuda.is_available() else "cpu"
emb_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
print("loaded MiniLM on", device)

train_prompts = [d["prompt"] for d in train_docs]
t0 = time.time()
train_embs = emb_model.encode(
    train_prompts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
print(f"train embeddings: {train_embs.shape} in {time.time()-t0:.1f}s")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


loaded MiniLM on cuda


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

train embeddings: (2000, 384) in 0.7s


In [23]:
# encode test prompts
set_seed()
test_prompts = test["prompt"].astype(str).tolist()
t0 = time.time()
test_embs = emb_model.encode(
    test_prompts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
print(f"test embeddings: {test_embs.shape} in {time.time()-t0:.1f}s")

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

test embeddings: (500, 384) in 0.2s


In [24]:
# retrieve top-K nearest train docs per test row
K = 3
sims = test_embs @ train_embs.T
top_k = np.argsort(-sims, axis=1)[:, :K]
print(f"top-{K} retrieval done, shape {top_k.shape}")
print(f"mean top-1 sim: {sims[np.arange(len(test)), top_k[:,0]].mean():.3f}")
print(f"median top-1 sim: {np.median(sims[np.arange(len(test)), top_k[:,0]]):.3f}")

top-3 retrieval done, shape (500, 3)
mean top-1 sim: 0.986
median top-1 sim: 1.000


In [25]:

for i in [0, 1, 2]:
    print(f"\ntest row {i}:")
    print("  Q:", test_prompts[i][:120])
    for j, tid in enumerate(top_k[i]):
        print(f"  match {j+1} (sim={sims[i,tid]:.3f}): {train_docs[tid]['prompt'][:100]}")


test row 0:
  Q: Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in supersymmetric quant
  match 1 (sim=1.000): Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in 
  match 2 (sim=1.000): Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in 
  match 3 (sim=0.993): Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in 

test row 1:
  Q: What is the estimated redshift of CEERS-93316, a candidate high-redshift galaxy observed by the James Webb Space Telesco
  match 1 (sim=1.000): What is the estimated redshift of CEERS-93316, a candidate high-redshift galaxy observed by the Jame
  match 2 (sim=0.919): Pick the best possible answer: What is the estimated redshift of CEERS-93316, a candidate high-redsh
  match 3 (sim=0.915): Determine the correct option: What is the estimated redshift of CEERS-93316, a 

In [26]:
# fit tf-idf on both splits option texts and prompts
corpus = []
for L in OPTIONS:
    corpus.extend(train[L].astype(str).tolist())
    corpus.extend(test[L].astype(str).tolist())
corpus.extend(train["prompt"].astype(str).tolist())
corpus.extend(test["prompt"].astype(str).tolist())

vec = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=1,
    max_features=50_000,
    sublinear_tf=True,
    strip_accents="unicode",
)
vec.fit(corpus)
print(f"tf-idf vocab: {len(vec.vocabulary_):,}")

tf-idf vocab: 12,749


In [27]:
# helper for matching a reference text to one of the 5 options via tf-idf
def tfidf_best_letter(test_row, ref_text):
    opt_texts = [str(test_row[L]) for L in OPTIONS]
    q = vec.transform([str(ref_text)])
    C = vec.transform(opt_texts)
    sim = cosine_similarity(q, C).ravel()
    return OPTIONS[int(np.argmax(sim))]

## LLM inference

Prompt shows the retrieved similar questions with their correct answers, then asks Qwen to pick a letter for the test question. Using Qwen 2.5 3B - 1.5B kept making basic mistakes last time.

In [28]:
# load the language model
from transformers import AutoTokenizer, AutoModelForCausalLM

set_seed()
llm_name = "Qwen/Qwen2.5-3B-Instruct"
tok = AutoTokenizer.from_pretrained(llm_name)
llm = AutoModelForCausalLM.from_pretrained(
    llm_name,
    torch_dtype=torch.float16,
    device_map="auto",
)
llm.eval()
print("loaded", llm_name)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

loaded Qwen/Qwen2.5-3B-Instruct


In [29]:
# build a RAG prompt and get an answer letter
def build_prompt(test_row, retrieved_ids):
    ctx = ""
    for k, tid in enumerate(retrieved_ids):
        d = train_docs[tid]
        ctx += f"Example {k+1}:\n"
        ctx += f"Question: {d['prompt']}\n"
        ctx += f"Correct answer: {d['correct_text']}\n\n"
    opts = "\n".join([f"{L}. {test_row[L]}" for L in OPTIONS])
    return (
        f"Here are some similar solved questions for reference:\n\n{ctx}"
        f"Now answer this question by picking the correct option.\n\n"
        f"Question: {test_row['prompt']}\n\n"
        f"Options:\n{opts}\n\n"
        f"Reply with just the single letter (A, B, C, D or E):"
    )

def parse_letter(text):
    m = re.search(r"[A-E]", str(text))
    return m.group(0) if m else None

def answer_row(test_row, retrieved_ids):
    set_seed()  # fresh reseed before each generation for reproducibility
    prompt = build_prompt(test_row, retrieved_ids)
    inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=2048).to(llm.device)
    with torch.no_grad():
        out = llm.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            pad_token_id=tok.eos_token_id,
        )
    gen = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    letter = parse_letter(gen)
    return letter, gen

In [30]:
# quick test on the first few test rows
for i in range(3):
    row = test.iloc[i]
    ret = top_k[i]
    letter, raw = answer_row(row, ret)
    tf_letter = tfidf_best_letter(row, train_docs[ret[0]]["correct_text"])
    print(f"row {i}: model={letter!r} (raw {raw.strip()!r}), tf-idf={tf_letter}")

row 0: model='A' (raw 'A\nYou are an'), tf-idf=A
row 1: model='B' (raw 'B'), tf-idf=B
row 2: model='B' (raw 'B'), tf-idf=B


## Full inference over test set

For each test row: get the model's letter pick and the tf-idf letter pick. Use the model if it produced a valid letter, otherwise fall back to tf-idf. This way parse failures don't tank the score.

In [31]:
predictions = []
llm_valid = 0
llm_fallback = 0
agree = 0

t0 = time.time()
for i in range(len(test)):
    row = test.iloc[i]
    ret = top_k[i]

    tf_pick = tfidf_best_letter(row, train_docs[ret[0]]["correct_text"])

    try:
        llm_pick, _ = answer_row(row, ret)
    except Exception as e:
        print(f"row {i} llm error:", e)
        llm_pick = None

    if llm_pick in OPTIONS:
        pick = llm_pick
        llm_valid += 1
        if llm_pick == tf_pick:
            agree += 1
    else:
        pick = tf_pick
        llm_fallback += 1

    predictions.append(pick)

    if (i+1) % 50 == 0:
        elapsed = time.time() - t0
        eta = elapsed / (i+1) * (len(test) - i - 1)
        print(f"done {i+1}/{len(test)} - {elapsed:.0f}s elapsed, eta {eta:.0f}s")

print()
print(f"total time: {time.time()-t0:.0f}s")
print(f"llm valid outputs: {llm_valid}/{len(test)}")
print(f"llm-tfidf agreement: {agree}/{llm_valid} ({100*agree/max(llm_valid,1):.1f}%)")
print(f"used tf-idf fallback: {llm_fallback}")
print(f"rank-1 letter distribution: {dict(Counter(predictions))}")

done 50/500 - 17s elapsed, eta 152s
done 100/500 - 32s elapsed, eta 130s
done 150/500 - 47s elapsed, eta 110s
done 200/500 - 64s elapsed, eta 96s
done 250/500 - 81s elapsed, eta 81s
done 300/500 - 97s elapsed, eta 64s
done 350/500 - 112s elapsed, eta 48s
done 400/500 - 127s elapsed, eta 32s
done 450/500 - 143s elapsed, eta 16s
done 500/500 - 160s elapsed, eta 0s

total time: 160s
llm valid outputs: 500/500
llm-tfidf agreement: 488/500 (97.6%)
used tf-idf fallback: 0
rank-1 letter distribution: {'A': 84, 'B': 112, 'E': 86, 'C': 121, 'D': 97}


In [32]:
# build top-3 predictions
top3 = []
for i, r1 in enumerate(predictions):
    row = test.iloc[i]
    r1_text = str(row[r1])
    rest = [L for L in OPTIONS if L != r1]
    rest_texts = [str(row[L]) for L in rest]
    q = vec.transform([r1_text])
    C = vec.transform(rest_texts)
    sim = cosine_similarity(q, C).ravel()
    order = np.argsort(-sim)
    top3.append([r1, rest[order[0]], rest[order[1]]])

for i in range(5):
    print(f"row {i}: {' '.join(top3[i])}")

row 0: A D B
row 1: B A C
row 2: B E D
row 3: E C A
row 4: C A D


In [33]:
# build the submission csv
sub = pd.DataFrame({
    "ID": test["id"],
    "Prediction": [" ".join(t) for t in top3]
})

# sanity checks
assert len(sub) == len(test), f"got {len(sub)} rows, expected {len(test)}"
assert (sub["ID"].values == sample["ID"].values).all(), "ID order mismatch"
assert sub["Prediction"].str.split().apply(len).eq(3).all(), "not all rows have 3 labels"
assert sub["Prediction"].str.split().apply(
    lambda x: all(L in OPTIONS for L in x) and len(set(x)) == 3
).all(), "invalid letter or duplicate in prediction"

sub.to_csv("/kaggle/working/submission.csv", index=False)
print(f"saved submission.csv with {len(sub)} rows")
sub.head(10)

saved submission.csv with 500 rows


,ID,Prediction
0,1,A D B
1,2,B A C
2,3,B E D
3,4,E C A
4,5,C A D
5,6,D E A
6,7,E D C
7,8,B E A
8,9,C D E
9,10,B E A
